# 포트핏 1단계: 내 ETF의 '진짜' 구성 들여다보기

이 노트북은 **ETF 여러 개를 샀을 때, 실제로는 어떤 종목에 얼마나 투자하고 있는지** 계산해요.
포트핏 앱의 핵심 엔진을 처음으로 직접 만들어 보는 실습이에요.

### 이 노트북 사용법
1. 위에서부터 차례대로, 각 회색 코드 칸을 클릭하고 **Shift + Enter**를 누르면 실행돼요.
2. 실행하면 코드 칸 바로 아래에 결과가 나와요.
3. 회색 칸 위의 설명을 먼저 읽고 실행하세요. 모르는 줄은 그대로 복사해서 AI에게 "이 줄은 뭐야?"라고 물어보세요.

> ⚠️ 여기 들어간 ETF 구성 종목과 비중은 **연습용으로 만든 가상 숫자**예요. 실제 ETF 비중과 다릅니다. 계산 방법을 익힌 뒤 마지막 단계에서 실제 데이터로 바꿔요.

## 1. 도구 불러오기

`pandas`는 파이썬에서 **엑셀처럼 표를 다루는 도구**예요. `import pandas as pd`는 "pandas를 불러오고, 앞으로 `pd`라는 짧은 이름으로 부를게"라는 뜻이에요.

In [ ]:
import pandas as pd

print("준비 완료!")

## 2. ETF 구성 종목 표 만들기

ETF 하나는 여러 종목을 담은 바구니예요. 아래 표는 ETF 4개가 각각 어떤 종목을 몇 %씩 담고 있는지 적은 거예요.

- 한 줄 = `(ETF 이름, 종목, 비중(%), 섹터, 국가)`
- `기타_...` 줄은 비중이 작은 나머지 종목들을 하나로 묶은 거예요. 그래야 ETF마다 합계가 100%가 돼요.

In [ ]:
holdings = pd.DataFrame([
    # (ETF 이름,          종목,            비중(%), 섹터,     국가)
    ("코스피200(예시)",    "삼성전자",        30, "반도체",   "한국"),
    ("코스피200(예시)",    "SK하이닉스",      12, "반도체",   "한국"),
    ("코스피200(예시)",    "LG에너지솔루션",   4, "2차전지",  "한국"),
    ("코스피200(예시)",    "현대차",           3, "자동차",   "한국"),
    ("코스피200(예시)",    "기타_국내",       51, "기타",     "한국"),

    ("반도체TOP10(예시)",  "SK하이닉스",      25, "반도체",   "한국"),
    ("반도체TOP10(예시)",  "삼성전자",        25, "반도체",   "한국"),
    ("반도체TOP10(예시)",  "한미반도체",      10, "반도체",   "한국"),
    ("반도체TOP10(예시)",  "기타_반도체",     40, "반도체",   "한국"),

    ("미국S&P500(예시)",   "엔비디아",         7, "반도체",   "미국"),
    ("미국S&P500(예시)",   "애플",             6, "IT하드웨어", "미국"),
    ("미국S&P500(예시)",   "마이크로소프트",   6, "소프트웨어", "미국"),
    ("미국S&P500(예시)",   "아마존",           4, "소비재",   "미국"),
    ("미국S&P500(예시)",   "기타_미국",       77, "기타",     "미국"),

    ("미국나스닥100(예시)", "엔비디아",         9, "반도체",   "미국"),
    ("미국나스닥100(예시)", "애플",             8, "IT하드웨어", "미국"),
    ("미국나스닥100(예시)", "마이크로소프트",   8, "소프트웨어", "미국"),
    ("미국나스닥100(예시)", "아마존",           5, "소비재",   "미국"),
    ("미국나스닥100(예시)", "기타_미국기술",   70, "기타",     "미국"),
], columns=["ETF", "종목", "비중", "섹터", "국가"])

holdings

### 검산: ETF마다 비중 합계가 100인지 확인

`groupby("ETF")`는 엑셀의 **피벗 테이블**과 같아요. ETF별로 묶어서 `비중`을 더해요(`sum`). 모두 100이 나와야 정상이에요.

In [ ]:
holdings.groupby("ETF")["비중"].sum()

## 3. 내 포트폴리오 입력하기

내가 각 ETF를 얼마나 샀는지 적어요(단위: 만 원). **이 숫자를 바꿔 보면서 결과가 어떻게 달라지는지 보는 게 이 실습의 핵심이에요.**

`계좌비중`은 "내 계좌 전체에서 이 ETF가 차지하는 비율"이에요. 금액 ÷ 총금액으로 구해요.

In [ ]:
my_portfolio = pd.DataFrame({
    "ETF":     ["코스피200(예시)", "반도체TOP10(예시)", "미국S&P500(예시)", "미국나스닥100(예시)"],
    "금액_만원": [100,               100,                150,                150],
})

my_portfolio["계좌비중"] = my_portfolio["금액_만원"] / my_portfolio["금액_만원"].sum()

my_portfolio

## 4. 실질 노출 계산하기 ⭐ 핵심

**공식:** 종목의 실질 비중 = Σ (내 계좌에서 ETF 비중 × 그 ETF 안에서 종목 비중)

예를 들어 삼성전자는:
- 코스피200 ETF 계좌비중 20% × ETF 안 삼성전자 30% = 6%
- 반도체TOP10 ETF 계좌비중 20% × ETF 안 삼성전자 25% = 5%
- 합계 = **11%**

코드로는 세 단계예요.
1. `merge`: 구성 종목 표와 내 포트폴리오 표를 ETF 이름으로 붙여요. 엑셀의 **VLOOKUP**과 같아요.
2. 곱하기: 각 줄마다 `비중 × 계좌비중`을 계산해요.
3. `groupby("종목").sum()`: 같은 종목끼리 더해요. 여러 ETF에 흩어진 삼성전자가 하나로 합쳐져요.

In [ ]:
merged = holdings.merge(my_portfolio, on="ETF")
merged["실질비중"] = merged["비중"] * merged["계좌비중"]

merged[["ETF", "종목", "비중", "계좌비중", "실질비중"]].head(8)

In [ ]:
exposure = merged.groupby("종목")["실질비중"].sum().sort_values(ascending=False).round(2)

# '기타_...'는 여러 종목을 묶은 것이라 개별 종목 순위에서는 빼요
exposure_named = exposure[~exposure.index.str.startswith("기타")]

print("검산 - 전체 합계(100이어야 정상):", round(exposure.sum(), 2))
exposure_named

## 5. 섹터·국가별 노출

같은 방법으로 `종목` 대신 `섹터`, `국가`로 묶으면 돼요. ETF 4개로 나눠 샀는데 **반도체 비중이 얼마나 되는지** 확인해 보세요.

In [ ]:
sector = merged.groupby("섹터")["실질비중"].sum().sort_values(ascending=False).round(1)
country = merged.groupby("국가")["실질비중"].sum().sort_values(ascending=False).round(1)

print("[섹터별 실질 비중 %]")
print(sector)
print()
print("[국가별 실질 비중 %]")
print(country)

## 6. ETF끼리 얼마나 겹치는지 (중복률)

**공식:** 두 ETF의 중복률 = 두 ETF에 모두 있는 종목마다 **더 작은 비중**을 골라 더한 값

예: S&P500에 엔비디아 7%, 나스닥100에 엔비디아 9% → 겹치는 부분은 7%

`def`는 **함수**를 만드는 명령이에요. 한 번 만들어 두면 `overlap("A", "B")`처럼 언제든 다시 쓸 수 있어요.

In [ ]:
def overlap(etf_a, etf_b):
    # 각 ETF의 종목별 비중 (기타 묶음은 제외)
    a = holdings[(holdings["ETF"] == etf_a) & (~holdings["종목"].str.startswith("기타"))].set_index("종목")["비중"]
    b = holdings[(holdings["ETF"] == etf_b) & (~holdings["종목"].str.startswith("기타"))].set_index("종목")["비중"]

    common = a.index.intersection(b.index)          # 두 ETF에 모두 있는 종목
    return sum(min(a[s], b[s]) for s in common)     # 더 작은 비중끼리 더하기


etfs = my_portfolio["ETF"].tolist()
rows = []
for i in range(len(etfs)):
    for j in range(i + 1, len(etfs)):               # 모든 ETF 쌍을 한 번씩
        rows.append((etfs[i], etfs[j], overlap(etfs[i], etfs[j])))

overlap_table = pd.DataFrame(rows, columns=["ETF 1", "ETF 2", "중복률(%)"])
overlap_table = overlap_table.sort_values("중복률(%)", ascending=False).reset_index(drop=True)
overlap_table

## 7. 집중도 (HHI)

**HHI(허핀달 지수)** = 각 종목 비중(%)을 제곱해서 모두 더한 값이에요. 경영학 산업조직론에서 시장 집중도를 잴 때 쓰는 그 지수예요.
- 몇 종목에 몰릴수록 숫자가 커지고, 고르게 퍼질수록 작아져요.
- 여기서는 `기타` 묶음을 빼고 계산했기 때문에 실제보다 조금 낮게 나와요. 실제 데이터로 바꾸면 정확해져요.

In [ ]:
hhi = (exposure_named ** 2).sum()
top3 = exposure_named.head(3).sum()

print(f"HHI (기타 제외): {hhi:.0f}")
print(f"상위 3종목 실질 비중 합계: {top3:.1f}%")

## 8. 차트로 보기

`plotly`는 차트를 그리는 도구예요. 구글 코랩에는 이미 설치되어 있어요. 차트에 마우스를 올리면 숫자가 보여요.

In [ ]:
import plotly.express as px

fig = px.bar(
    exposure_named.reset_index(),
    x="실질비중", y="종목", orientation="h",
    title="내 계좌의 실질 종목 노출 (%)",
)
fig.update_layout(yaxis={"categoryorder": "total ascending"})
fig.show()

In [ ]:
fig2 = px.pie(sector.reset_index(), names="섹터", values="실질비중", title="섹터별 실질 비중")
fig2.show()

## 9. 진단 결과 요약

앱에서 사용자에게 보여줄 문장을 만들어 봐요. `f"...{변수}..."`는 문장 안에 계산 결과를 끼워 넣는 방법이에요.

In [ ]:
top_pair = overlap_table.iloc[0]

print(f"■ 내 ETF {len(etfs)}개 진단 결과")
print(f"- 가장 많이 담긴 종목: {exposure_named.index[0]} {exposure_named.iloc[0]:.1f}%")
print(f"- 반도체 섹터 비중: {sector.get('반도체', 0):.1f}%")
print(f"- 미국 비중: {country.get('미국', 0):.1f}%")
print(f"- 가장 많이 겹치는 ETF: {top_pair['ETF 1']} ↔ {top_pair['ETF 2']} (중복률 {top_pair['중복률(%)']:.0f}%)")

## 10. 연습 과제

1. **금액 바꿔보기:** 3번 칸에서 `반도체TOP10(예시)` 금액을 300으로 바꾸고, 3번부터 아래로 다시 실행해 보세요. 반도체 비중이 몇 %가 되나요?
2. **ETF 빼보기:** `미국나스닥100(예시)`를 포트폴리오에서 지우면 중복률 표가 어떻게 바뀌나요?
3. **손으로 검산하기:** SK하이닉스의 실질 비중을 계산기로 직접 구하고, 4번 결과와 맞는지 확인해 보세요.
4. **다음 단계 준비:** 실제 ETF 구성 종목은 한국거래소 정보데이터시스템이나 운용사(KODEX, TIGER) 홈페이지에서 파일로 받을 수 있어요. 내가 가진 ETF 하나의 구성 종목 파일을 내려받아 두세요. 다음 실습에서 2번 칸의 가상 데이터를 이 파일로 바꿀 거예요.